In [11]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="gpt-oss:120b-cloud")
print('Model is ready to use!')

Model is ready to use!


In [12]:
from langchain_community.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get weather for a city"""
    return f"The weather in the city {city} is Sunny"

@tool
def get_best_stock() -> str:
    """Returns the best stock to invest in"""
    return f"The best stock to invest in is Zensar"

@tool
def purchase_stock(stock: str, quantity: int) -> str:
    """Purchases a given stock for the specified quantity"""
    return f"Purchased {quantity} shares of {stock}"

ALL_TOOLS = [get_weather, get_best_stock, purchase_stock]
SENSITIVE_TOOLS = {"purchase_stock"}

print('Tools are are ready')

Tools are are ready


In [13]:
# STEP-1: Define the state
from typing import TypedDict, Annotated
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

print('Graph state is ready .. ')

Graph state is ready .. 


In [14]:
# STEP-2: Define the nodes
from langchain_core.messages import SystemMessage, ToolMessage
from langgraph.types import Command, interrupt

model_with_tools = model.bind_tools(ALL_TOOLS)

HITL_SYSTEMTPROMPT = SystemMessage(content=(
    "You have access to get_weather, get_best_stock, and purchase_stock tools. "
    "When the user asks to find best stock , immediately call the get_best_stock tool. "
    "When the user asks to purchase stock, immediately call the purchase_stock "
    "tool with the requested stock and quantity - do not ask the user to confirm "
    "in plain text first. A human reviewer approves or denies the purchase_stock tool call"
    "automatically before it executes, so your job is only to call the tool."
))

def assistant(state: AgentState) -> dict:
    """One graph node: call the model using current messages"""
    response = model_with_tools.invoke([HITL_SYSTEMTPROMPT] + state['messages'])
    return {"messages": [response]}

def route_after_assistant(state: AgentState) -> str:
    """Route to the tool only when model produces tool calls"""
    last_message = state['messages'][-1]
    tool_calls = getattr(last_message, "tool_calls", None)
    if not tool_calls:
        return "end"
    if any(tool_call['name'] in SENSITIVE_TOOLS for tool_call in tool_calls):
        return "human_review"
    return "tools"

def human_review(state: AgentState) -> str:
    """Human review node: asks for human approval"""
    last_message = state["messages"][-1]
    tool_call = last_message.tool_calls[0]
    
    # Pause the graph execution for human review and wait for approval
    decision = interrupt({
        "question": "Shall I really purchase this stock?",
        "tool": tool_call['name'],
        "args": tool_call['args']
    })
    
    if decision == "approve":
        return Command(goto="tools")
    
    denial = ToolMessage(
        content=f"Human has rejected purchasing this stock",
        tool_call_id=tool_call['id']
    )
    
    return Command(goto="assistant", update={"messages": [denial]})
    
print('Graph Nodes are ready')

Graph Nodes are ready


In [15]:
# STEP-3: Add edges and build the graph
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

from langgraph.checkpoint.memory import MemorySaver
# from langgraph.checkpoint.sqlite import SqliteSaver
# import sqlite3

short_term_memory = MemorySaver()
# short_term_memory = SqliteSaver(sqllite_connection)

graph_builder = StateGraph(AgentState)

# Node registration
graph_builder.add_node("assistant", assistant)
graph_builder.add_node("human_review", human_review)
graph_builder.add_node("tools", ToolNode(ALL_TOOLS))

# Add edges
graph_builder.add_edge(START, "assistant")
graph_builder.add_conditional_edges(
    "assistant",
    route_after_assistant,
    {
        "human_review": "human_review",
        "tools": "tools",
        "end": END
    }
)
graph_builder.add_edge("tools", "assistant")
graph_builder.add_edge("assistant", END)

graph = graph_builder.compile(checkpointer=short_term_memory)
print('Graph is ready ... ')
# graph

Graph is ready ... 


In [16]:
hitl_config = {"configurable": {"thread_id": "stock_session"}}
response = graph.invoke({
    "messages": [
        {"role": "user", "content": "Which is the best stock to invest in?"}
    ]
}, hitl_config)

if "_interrupt_" in response:
    print("Graph execution paused for human review ... ")
    print(response["_interrupt_"])
else:
    print(f'Graph execution completed without human review.')
    print(response)

# response

Graph execution completed without human review.
{'messages': [HumanMessage(content='Which is the best stock to invest in?', additional_kwargs={}, response_metadata={}, id='f466cb62-b3de-42b2-b939-73bebc44ed24'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-11-03T09:51:01.623146718Z', 'done': True, 'done_reason': 'stop', 'total_duration': 369910069, 'load_duration': None, 'prompt_eval_count': 274, 'prompt_eval_duration': None, 'eval_count': 59, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'}, id='lc_run--01a1a0d1-b79e-73f6-9be5-4114bfeeb31f-0', tool_calls=[{'name': 'get_best_stock', 'args': {}, 'id': 'cedfeedb-1f4a-4c06-911e-acfa2c05059b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 274, 'output_tokens': 59, 'total_tokens': 333}), ToolMessage(content='The best stock to invest in is Zensar', name='get_best_stock', id='b3e7d0e5-5753-4d16-b926-

In [19]:
hitl_config = {"configurable": {"thread_id": "stock_session"}}
response = graph.invoke({
    "messages": [
        {"role": "user", "content": "Please purchase Zensar stocks for 10 shares."}
    ]
}, hitl_config)

if "_interrupt_" in response:
    print("Graph execution paused for human review ... ")
    print(response["_interrupt_"])
else:
    print(f'Graph execution completed without human review.')
    print(response)

# response
# result is not coming similar to what Anand - the trainer got in his code

Graph execution completed without human review.
{'messages': [HumanMessage(content='Which is the best stock to invest in?', additional_kwargs={}, response_metadata={}, id='f466cb62-b3de-42b2-b939-73bebc44ed24'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-11-03T09:51:01.623146718Z', 'done': True, 'done_reason': 'stop', 'total_duration': 369910069, 'load_duration': None, 'prompt_eval_count': 274, 'prompt_eval_duration': None, 'eval_count': 59, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'}, id='lc_run--01a1a0d1-b79e-73f6-9be5-4114bfeeb31f-0', tool_calls=[{'name': 'get_best_stock', 'args': {}, 'id': 'cedfeedb-1f4a-4c06-911e-acfa2c05059b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 274, 'output_tokens': 59, 'total_tokens': 333}), ToolMessage(content='The best stock to invest in is Zensar', name='get_best_stock', id='b3e7d0e5-5753-4d16-b926-

In [18]:
approve_command = Command(resume="approve")
approve_response = graph.invoke(approve_command, hitl_config)
approve_response

{'messages': [HumanMessage(content='Which is the best stock to invest in?', additional_kwargs={}, response_metadata={}, id='f466cb62-b3de-42b2-b939-73bebc44ed24'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-11-03T09:51:01.623146718Z', 'done': True, 'done_reason': 'stop', 'total_duration': 369910069, 'load_duration': None, 'prompt_eval_count': 274, 'prompt_eval_duration': None, 'eval_count': 59, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'}, id='lc_run--01a1a0d1-b79e-73f6-9be5-4114bfeeb31f-0', tool_calls=[{'name': 'get_best_stock', 'args': {}, 'id': 'cedfeedb-1f4a-4c06-911e-acfa2c05059b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 274, 'output_tokens': 59, 'total_tokens': 333}),
  ToolMessage(content='The best stock to invest in is Zensar', name='get_best_stock', id='b3e7d0e5-5753-4d16-b926-1fdf4a6365b7', tool_call_id='cedfeedb-1f4a-4

In [20]:
deny_command = Command(resume="deny")
deny_response = graph.invoke(deny_command, hitl_config)
approve_response

{'messages': [HumanMessage(content='Which is the best stock to invest in?', additional_kwargs={}, response_metadata={}, id='f466cb62-b3de-42b2-b939-73bebc44ed24'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-11-03T09:51:01.623146718Z', 'done': True, 'done_reason': 'stop', 'total_duration': 369910069, 'load_duration': None, 'prompt_eval_count': 274, 'prompt_eval_duration': None, 'eval_count': 59, 'eval_duration': None, 'logprobs': None, 'model_name': 'gpt-oss:120b', 'model_provider': 'ollama'}, id='lc_run--01a1a0d1-b79e-73f6-9be5-4114bfeeb31f-0', tool_calls=[{'name': 'get_best_stock', 'args': {}, 'id': 'cedfeedb-1f4a-4c06-911e-acfa2c05059b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 274, 'output_tokens': 59, 'total_tokens': 333}),
  ToolMessage(content='The best stock to invest in is Zensar', name='get_best_stock', id='b3e7d0e5-5753-4d16-b926-1fdf4a6365b7', tool_call_id='cedfeedb-1f4a-4